# Week 7: Local RAG Document Question Answering System

**Name**: Khushi Bhagat  
**Project**: Building a Retrieval-Augmented Generation (RAG) chatbot using a Local TF-IDF Vectorizer & Cosine Similarity, and Groq API.

## What is RAG?
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models to answer questions based on custom documents (like PDFs). This keeps the answers factual and grounded in the source material.

## Pipeline Steps
1. **Document Loading**: Extracting raw text from a PDF document using PyMuPDF.
2. **Text Chunking**: Splitting the raw text into smaller, overlapping segments.
3. **Local Vector Representations**: Converting text chunks into TF-IDF vectors using `scikit-learn` fully locally (requires no cloud setup and runs instantly).
4. **Vector Similarity Search**: Calculating Cosine Similarity between the query vector and document vectors to retrieve the most relevant text segments.
5. **Grounded Generation**: Sending the retrieved context and question to Groq's fast Llama 3 model to generate the final response.

Let's build and examine this pipeline step-by-step!

## Step 0: Imports and Configuration
First, we import the necessary modules. We use PyMuPDF (`fitz`), `scikit-learn` (`TfidfVectorizer`, `cosine_similarity`), and Groq SDK. We also use `dotenv` to load the Groq API key securely from a `.env` file.

In [ ]:
import os
import fitz  # PyMuPDF for loading PDFs
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
from dotenv import load_dotenv
import getpass

# Load environment variables from .env file
load_dotenv()

print("All libraries imported successfully!")

## Step 1: PDF Text Extraction
We read the PDF file page-by-page and extract raw text using PyMuPDF (`fitz`).

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Reads a PDF file and extracts all its raw text"""
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page_num in range(pdf.page_count):
            page = pdf.load_page(page_num)
            text += page.get_text("text")
    return text

# Path to the demonstration PDF file
pdf_path = r"C:\Users\khush\YTRAG\data\PDF\Embeddings.pdf"
raw_text = extract_text_from_pdf(pdf_path)
print(f"Loaded PDF text: {len(raw_text)} characters extracted.")
print(f"Preview (first 300 chars):\n---\n{raw_text[:300]}\n---")

## Step 2: Text Chunking
Since PDFs can contain massive amounts of text, we must split it into smaller overlapping segments (chunks). We split by sentences to keep content intact.

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=100):
    """Splits text into smaller chunks by sentence boundaries"""
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip() + ".")
            # Simple overlap logic: carry forward a small part of the previous chunk
            words = current_chunk.split()
            overlap_words = words[-max(1, int(overlap/6)):] if len(words) > 10 else []
            current_chunk = " ".join(overlap_words) + " " + sentence + ". "
        else:
            current_chunk += sentence + ". "
            
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

text_chunks = chunk_text(raw_text)
print(f"Split document into {len(text_chunks)} semantic chunks.")
print(f"Preview of Chunk 1 (length {len(text_chunks[0])}):\n---\n{text_chunks[0]}\n---")

## Step 3: Local TF-IDF Vectorization
We convert the text chunks into mathematical vectors. We fit a TF-IDF (Term Frequency-Inverse Document Frequency) model on our text chunks to establish a vocabulary space.

In [ ]:
print("Initializing TF-IDF Vectorizer locally...")
vectorizer = TfidfVectorizer()

# Fit the vectorizer on all text chunks and transform them into dense vectors
tfidf_matrix = vectorizer.fit_transform(text_chunks)
print(f"Vectorization complete. Matrix shape: {tfidf_matrix.shape} (chunks, vocabulary size)")
print(f"Vocabulary size (number of unique terms): {len(vectorizer.vocabulary_)}")

## Step 4: Similarity Query and Retrieval
When a user asks a question, we convert the query into the same TF-IDF space and compute Cosine Similarity to find the most relevant chunks.

In [ ]:
def retrieve_context(query, vectorizer, tfidf_matrix, chunks, top_k=3):
    """Calculates cosine similarity and retrieves the most relevant chunks"""
    query_vec = vectorizer.transform([query])
    
    # Calculate cosine similarity between the query and all indexed chunks
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get indices of the top_k most similar chunks
    top_indices = similarities.argsort()[::-1][:top_k]
    
    retrieved = []
    for idx in top_indices:
        if similarities[idx] > 0.0:
            retrieved.append(chunks[idx])
            print(f"Found match at chunk index {idx} (Similarity score: {similarities[idx]:.3f})")
    
    # Fallback if no relevant chunks found
    if not retrieved and chunks:
        retrieved = chunks[:top_k]
        
    return retrieved

test_query = "What are embeddings and why are they used?"
retrieved_context = retrieve_context(test_query, vectorizer, tfidf_matrix, text_chunks)
print(f"\nQuery: '{test_query}'")
print(f"Retrieved {len(retrieved_context)} relevant chunks.")

## Step 5: Groq LLM Answer Generation
We send the retrieved context blocks along with the user's query to Groq's fast LLM (Llama 3) to generate a grounded, fact-checked response. 

**Key Security**: To prevent key leaks, we check for a loaded environment variable. If missing, we prompt the user securely at runtime using `getpass` so that keys are never stored in the notebook code or output JSON.

In [ ]:
# Retrieve Groq key securely
groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    groq_key = getpass.getpass("Groq API Key not found in environment. Please enter it: ")

# Initialize Groq client
groq_client = Groq(api_key=groq_key)

def generate_grounded_answer(query, context_chunks, client):
    """Generates a fact-grounded response using context chunks and Groq LLM"""
    context_text = "\n\n".join([f"[Source {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)])
    
    system_prompt = (
        "You are a document question-answering assistant. Answer the question using ONLY the provided context blocks. "
        "If the context does not contain the answer, state that you cannot find the answer in the document. "
        "Be extremely factual, clear, and concise. Do not guess."
    )
    
    user_prompt = f"Context blocks:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"
    
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

# Test generation
try:
    answer = generate_grounded_answer(test_query, retrieved_context, groq_client)
    print(f"Q: {test_query}")
    print(f"\nA: {answer}")
except Exception as e:
    print(f"Could not query Groq API (Check API key): {e}")

## Step 6: Packaging as Modular Classes
For cleaner codebase integration (and for our Streamlit app), we pack the logic into modular classes: `VectorStore` (in `vectorstore.py`) and `Chatbot` (in `chatbot.py`). Let's define them here.

In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.chunks = []
        self.tfidf_matrix = None

    def process_and_index(self, pdf_path):
        raw_text = extract_text_from_pdf(pdf_path)
        self.chunks = chunk_text(raw_text)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.chunks)
        return len(self.chunks)
        
    def retrieve(self, query, top_k=3):
        if not self.chunks or self.tfidf_matrix is None:
            return []
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        top_indices = similarities.argsort()[::-1][:top_k]
        retrieved_chunks = []
        for idx in top_indices:
            if similarities[idx] > 0.0:
                retrieved_chunks.append(self.chunks[idx])
        if not retrieved_chunks and self.chunks:
            retrieved_chunks = self.chunks[:top_k]
        return retrieved_chunks

class SimpleChatbot:
    def __init__(self, api_key):
        self.client = Groq(api_key=api_key)
        
    def respond(self, query, context_chunks):
        return generate_grounded_answer(query, context_chunks, self.client)

print("Modular wrapper classes defined successfully!")

## Step 7: Streamlit Application Walkthrough
We have designed a Streamlit application in the `src/` folder that builds a premium frontend for this RAG bot. It contains:
- Sidebar to enter API keys securely (never exposed to public code).
- File uploader allowing drag-and-drop of any PDF document.
- Message logging session state to render a standard, chat room interface.

To run the Streamlit app locally:
```bash
cd week7_rag_project/src
streamlit run app.py
```

## Key Learnings & Conclusion
- **Modularity**: Keeping components separated in cells makes scaling and debugging straightforward.
- **Local Retrieval**: TF-IDF cosine similarity search allows high performance, local operation without database hosting costs, downloads, or memory crashes.
- **API Security**: Using `.env` environment variables or python's `getpass` runtime prompt secures keys against repository leakage.